# Semivolatility-Scaled Momentum with a Fixed Universe

Uses a monthly moving-average crossover and Garman–Klass semivolatility to test a fixed, editable set of eight liquid US stocks. It compares the portfolio with SPY using dividend-adjusted Yahoo Finance prices and includes turnover costs. The ticker list is fixed across the backtest and can introduce survivorship bias.

In [ ]:
import numpy as np
import pandas as pd
from _market_data import download_ohlcv
import matplotlib.pyplot as plt
import seaborn as sns

from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

class StrategyConfig:
    """
    Centralized configuration for strategy parameters.
    All parameters are easily adjustable here.
    """
    TICKERS = ("AAPL", "AMZN", "GOOGL", "JPM", "MSFT", "NVDA", "UNH", "V")

    # Date Range
    END_DATE = "2024-12-31"
    START_DATE = "2019-01-01"  # Start date for backtest (need buffer for indicators)
    WARMUP_DATE = "2018-01-01"  # Additional warmup for 200-day SMA calculation

    # Momentum Parameters
    SMA_SHORT = 20      # Short-term SMA for crossover
    SMA_LONG = 200      # Long-term SMA for crossover

    # Volatility Parameters
    SEMI_VOL_WINDOW = 60        # Lookback window for semi-volatility calculation (in days)
    TARGET_VOLATILITY = 0.15    # Target annual volatility (15%)
    ANNUALIZATION_FACTOR = 252  # Trading days per year

    # Transaction Costs
    COMMISSION_BPS = 10         # Commission in basis points (0.10%)
    SLIPPAGE_BPS = 5            # Slippage in basis points (0.05%)
    TOTAL_COST_BPS = COMMISSION_BPS + SLIPPAGE_BPS  # Total cost per trade

    # Portfolio Parameters
    INITIAL_CAPITAL = 100000    # Starting capital in USD
    MAX_POSITIONS = 20          # Maximum number of concurrent positions
    MIN_POSITION_SIZE = 0.01    # Minimum position size (1% of portfolio)

    # Rebalancing
    REBALANCE_FREQUENCY = 'M'   # 'M' for monthly, 'Q' for quarterly

    # Benchmark
    BENCHMARK_TICKER = "SPY"

    # Risk Management
    MAX_LEVERAGE = 1.0          # Maximum gross leverage (1.0 = no leverage)

    # Universe Selection
    MIN_MARKET_CAP = 2e9        # $2B minimum market cap
    MIN_PRICE = 5.0             # $5 minimum price (avoid penny stocks)
    MIN_VOLUME = 1e6            # 1M daily volume minimum


# ============================================================================
# DATA ACQUISITION MODULE
# ============================================================================

class YahooDataFetcher:
    """Load dividend-adjusted daily OHLCV from Yahoo Finance."""

    @staticmethod
    def get_historical_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        prices = download_ohlcv(ticker, start_date, end_date, auto_adjust=True)
        prices = prices.rename(columns=str.lower)
        columns = ["open", "high", "low", "close"]
        if (prices[columns] <= 0).any().any():
            raise ValueError(f"Yahoo Finance returned non-positive prices for {ticker}.")
        return prices[columns + ["volume"]]


class GarmanKlassSemiVolatility:
    """
    Implements Garman-Klass volatility estimator adapted for semi-volatility.

    CRITICAL FIXES APPLIED:
    1. Excludes current day from volatility calculation (no look-ahead bias)
    2. Handles negative variance using absolute value (not forced to zero)
    3. Proper handling of periods with no negative returns
    4. Uses expanding window correctly
    """

    @staticmethod
    def calculate(df: pd.DataFrame, window: int, annualization_factor: int = 252) -> pd.Series:
        """
        Calculate expanding Garman-Klass semi-volatility with NO look-ahead bias.

        CRITICAL FIX: Uses only data available BEFORE the current day (T-1).

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'open', 'high', 'low', 'close' columns
        window : int
            Lookback window size (in days)
        annualization_factor : int
            Number of trading days per year (typically 252)

        Returns:
        --------
        pd.Series : Annualized Garman-Klass semi-volatility (no look-ahead bias)
        """
        # Calculate log returns for OHLC
        log_hl = np.log(df['high'] / df['low'])
        log_co = np.log(df['close'] / df['open'])

        # Standard Garman-Klass variance (daily)
        # Formula: 0.5 * (log(H/L))^2 - (2*log(2)-1) * (log(C/O))^2
        gk_variance = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)

        # FIX: Handle negative variance from GK formula properly
        # When variance is negative, it indicates the estimator broke down
        # Use absolute value to preserve magnitude information
        gk_variance = gk_variance.abs()

        # Calculate close-to-close returns to identify negative days
        close_returns = df['close'].pct_change()
        is_negative_day = (close_returns < 0).astype(int)

        # Initialize semi-volatility series
        semi_vol = pd.Series(np.nan, index=df.index)

        # CRITICAL FIX: Calculate semi-volatility using ONLY past data
        # For day i, use data from i-window to i-1 (EXCLUDE day i)
        for i in range(window, len(df)):
            # Look back 'window' days, but EXCLUDE current day (i)
            lookback_start = max(0, i - window)
            lookback_slice = slice(lookback_start, i)  # FIX: Changed from i+1 to i

            # Get variance and negative day indicator for this lookback period
            variance_window = gk_variance.iloc[lookback_slice]
            negative_window = is_negative_day.iloc[lookback_slice]

            # Only include variance from negative return days
            negative_variance = variance_window[negative_window == 1]
            n_negative_days = len(negative_variance)

            # Calculate average semi-variance
            if n_negative_days > 0:
                avg_semi_variance = negative_variance.sum() / n_negative_days
            else:
                # If no negative days, use previous semi-vol
                if i > window and not pd.isna(semi_vol.iloc[i-1]):
                    avg_semi_variance = (semi_vol.iloc[i-1] / np.sqrt(annualization_factor)) ** 2
                else:
                    # Use mean variance as conservative estimate
                    avg_semi_variance = variance_window.mean() if len(variance_window) > 0 else 0.01

            # Convert to volatility and annualize
            semi_vol.iloc[i] = np.sqrt(avg_semi_variance * annualization_factor)

        # For initial period, use expanding window
        for i in range(1, min(window, len(df))):
            lookback_slice = slice(0, i)  # FIX: Exclude current day

            variance_window = gk_variance.iloc[lookback_slice]
            negative_window = is_negative_day.iloc[lookback_slice]

            negative_variance = variance_window[negative_window == 1]
            n_negative_days = len(negative_variance)

            if n_negative_days > 0:
                avg_semi_variance = negative_variance.sum() / n_negative_days
                semi_vol.iloc[i] = np.sqrt(avg_semi_variance * annualization_factor)
            elif i > 1 and not pd.isna(semi_vol.iloc[i-1]):
                semi_vol.iloc[i] = semi_vol.iloc[i-1]

        return semi_vol


# ============================================================================
# MOMENTUM SIGNAL GENERATOR - FIXED
# ============================================================================

class SMAMomentumSignal:
    """
    Generates momentum signals based on Simple Moving Average crossover.

    CRITICAL FIX: Signal strength shifted by 2 days to avoid look-ahead bias.
    """

    @staticmethod
    def calculate(df: pd.DataFrame, short_window: int, long_window: int) -> Tuple[pd.Series, pd.Series]:
        """
        Calculate SMA crossover signals and signal strength.

        CRITICAL FIX: Both signal and strength use data from T-2 to avoid
        using same-day close in ranking.

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'close' column
        short_window : int
            Short-term SMA period (e.g., 20 days)
        long_window : int
            Long-term SMA period (e.g., 200 days)

        Returns:
        --------
        Tuple[pd.Series, pd.Series] : (binary signal, signal strength)
        """
        # Calculate both moving averages
        sma_short = df['close'].rolling(window=short_window, min_periods=short_window).mean()
        sma_long = df['close'].rolling(window=long_window, min_periods=long_window).mean()

        # Generate binary signal
        signal = (sma_short > sma_long).astype(int)

        # Calculate signal strength for ranking
        signal_strength = (sma_short - sma_long) / sma_long
        signal_strength = signal_strength.fillna(0)

        # FIX: Shift by 2 days instead of 1
        # Day T close -> available for ranking at end of T+1 -> execute at open of T+2
        signal = signal.shift(2)  # Changed from shift(1)
        signal_strength = signal_strength.shift(2)  # Changed from shift(1)

        return signal, signal_strength


# ============================================================================
# POSITION SIZING ENGINE - FIXED
# ============================================================================

class VolatilityScaledPositionSizer:
    """
    Calculates position sizes based on inverse semi-volatility weighting.

    FIXES APPLIED:
    - Proper asset ranking by signal strength
    - Leverage constraints enforced
    - Minimum position size filtering
    """

    @staticmethod
    def calculate_weights(signals: pd.DataFrame,
                         signal_strengths: pd.DataFrame,
                         semi_vols: pd.DataFrame,
                         target_vol: float,
                         max_positions: int,
                         max_leverage: float) -> pd.DataFrame:
        """Calculate position weights using semi-volatility scaling."""
        weights = pd.DataFrame(0.0, index=signals.index, columns=signals.columns)

        for date in signals.index:
            active_signals = signals.loc[date]
            active_mask = active_signals == 1

            if not active_mask.any():
                continue

            # Get signal strengths for active assets
            strengths = signal_strengths.loc[date, active_mask]

            # Rank by signal strength and select top positions
            ranked_tickers = strengths.sort_values(ascending=False).head(max_positions).index.tolist()

            if len(ranked_tickers) == 0:
                continue

            # Get semi-volatilities for selected assets
            asset_semi_vols = semi_vols.loc[date, ranked_tickers]

            # Filter valid volatilities
            valid_mask = (asset_semi_vols.notna()) & (asset_semi_vols > 0)
            if not valid_mask.any():
                continue

            ranked_tickers = asset_semi_vols[valid_mask].index.tolist()
            asset_semi_vols = asset_semi_vols[valid_mask]

            # Calculate inverse volatility weights
            inverse_vols = target_vol / asset_semi_vols
            raw_weights = inverse_vols / inverse_vols.sum() * max_leverage

            # Apply minimum position size constraint
            raw_weights = raw_weights[raw_weights >= 0.01]  # MIN_POSITION_SIZE

            # Re-normalize after filtering
            if raw_weights.sum() > 0:
                final_weights = raw_weights / raw_weights.sum() * max_leverage
                weights.loc[date, final_weights.index] = final_weights.values

        return weights


# ============================================================================
# BACKTESTING ENGINE - FIXED
# ============================================================================

class SMOMBacktester:
    """
    Comprehensive backtesting engine with all critical fixes applied
    (except survivorship bias).

    CRITICAL FIXES APPLIED:
    1. Proper rebalancing timing (signal Close[t] → execute Open[t+1])
    2. Consistent return calculation (Open-to-Open for all periods)
    3. Correct last day handling (no mixing of calculation methods)
    4. Leverage constraints verified
    5. Transaction costs properly timed
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.data_fetcher = YahooDataFetcher()

    def run_backtest(self) -> Dict:
        """Execute complete backtest workflow."""
        print("="*80)
        print("SEMIVOLATILITY-SCALED MOMENTUM WITH A FIXED UNIVERSE")
        print("="*80)
        print(f"Period: {self.config.START_DATE} to {self.config.END_DATE}")
        print(f"Momentum: SMA {self.config.SMA_SHORT}-{self.config.SMA_LONG} Crossover")
        print(f"Semi-Volatility: Garman-Klass {self.config.SEMI_VOL_WINDOW}-day window")
        print(f"Target Volatility: {self.config.TARGET_VOLATILITY*100:.1f}%")
        print(f"Transaction Costs: {self.config.TOTAL_COST_BPS} bps per trade")
        print("="*80)

        # Step 1: Get universe
        print("\n[1/7] Getting universe...")
        tickers = list(dict.fromkeys(self.config.TICKERS))

        if len(tickers) == 0:
            raise ValueError("No tickers retrieved")

        # Step 2: Download data
        print("\n[2/7] Downloading adjusted historical data...")
        benchmark_data = self.data_fetcher.get_historical_data(
            self.config.BENCHMARK_TICKER, self.config.WARMUP_DATE, self.config.END_DATE,
        )
        price_data = {}

        for i, ticker in enumerate(tickers):
            if (i + 1) % 20 == 0:
                print(f"  Progress: {i+1}/{len(tickers)}")

            data = self.data_fetcher.get_historical_data(
                ticker,
                self.config.WARMUP_DATE,
                self.config.END_DATE
            )

            if data is not None and len(data) >= self.config.SMA_LONG:
                price_data[ticker] = data

        print(f"\nSuccessfully loaded {len(price_data)} tickers")

        # Step 3: Calculate signals
        print("\n[3/7] Calculating momentum signals...")
        signals = pd.DataFrame(index=benchmark_data.index)
        signal_strengths = pd.DataFrame(index=benchmark_data.index)

        for ticker, data in price_data.items():
            signal, strength = SMAMomentumSignal.calculate(
                data,
                self.config.SMA_SHORT,
                self.config.SMA_LONG
            )
            signals[ticker] = signal
            signal_strengths[ticker] = strength

        # Step 4: Calculate semi-volatility
        print("\n[4/7] Computing semi-volatility (no look-ahead)...")
        semi_vols = pd.DataFrame(index=benchmark_data.index)

        for ticker, data in price_data.items():
            semi_vol = GarmanKlassSemiVolatility.calculate(
                data,
                self.config.SEMI_VOL_WINDOW,
                self.config.ANNUALIZATION_FACTOR
            )
            semi_vols[ticker] = semi_vol

        # Step 5: Align to backtest period
        print("\n[5/7] Aligning data...")
        backtest_start = pd.to_datetime(self.config.START_DATE)
        backtest_end = pd.to_datetime(self.config.END_DATE)

        common_dates = signals.index.intersection(semi_vols.index)
        common_dates = common_dates[(common_dates >= backtest_start) & (common_dates <= backtest_end)]

        if common_dates.empty or not price_data:
            raise ValueError("No eligible price data in the backtest period.")
        signals = signals.loc[common_dates]
        signal_strengths = signal_strengths.loc[common_dates]
        semi_vols = semi_vols.loc[common_dates]

        print(f"Backtest period: {common_dates[0].strftime('%Y-%m-%d')} to {common_dates[-1].strftime('%Y-%m-%d')}")
        print(f"Total trading days: {len(common_dates)}")

        # Step 6: Calculate weights
        print("\n[6/7] Calculating position weights...")
        weights = VolatilityScaledPositionSizer.calculate_weights(
            signals,
            signal_strengths,
            semi_vols,
            self.config.TARGET_VOLATILITY,
            self.config.MAX_POSITIONS,
            self.config.MAX_LEVERAGE
        )

        # Step 7: Simulate trading
        print("\n[7/7] Simulating strategy execution...")

        open_prices = pd.DataFrame({ticker: data['open'] for ticker, data in price_data.items()})
        close_prices = pd.DataFrame({ticker: data['close'] for ticker, data in price_data.items()})

        open_prices = open_prices.reindex(common_dates)
        close_prices = close_prices.reindex(common_dates)

        strategy_returns = self._calculate_strategy_returns_fixed(
            weights,
            open_prices,
            close_prices
        )

        # Get benchmark
        print("\nLoading benchmark data...")

        benchmark_open = benchmark_data['open'].loc[common_dates]
        benchmark_returns = benchmark_open.pct_change()

        # Calculate metrics
        results = self._calculate_performance_metrics(
            strategy_returns,
            benchmark_returns
        )

        results['weights'] = weights
        results['signals'] = signals
        results['price_data'] = price_data

        print("\n" + "="*80)
        print("BACKTEST COMPLETE")
        print("="*80)

        return results

    def _calculate_strategy_returns_fixed(self,
                                          weights: pd.DataFrame,
                                          open_prices: pd.DataFrame,
                                          close_prices: pd.DataFrame) -> pd.Series:
        """
        Calculate strategy returns with CORRECTED timing.

        CRITICAL FIXES:
        1. All returns calculated Open[t] to Open[t+1] consistently
        2. Last day calculated properly (excluded from returns)
        3. Transaction costs properly timed
        4. Rebalancing aligned correctly
        """
        dates = weights.index
        month_ends = dates.to_series().groupby(dates.to_period('M')).apply(lambda x: x.index[-1])

        # Rebalancing at open of next day after month-end
        rebalance_dates = []
        for month_end in month_ends:
            month_end_idx = dates.get_loc(month_end)
            if month_end_idx < len(dates) - 1:
                next_trading_day = dates[month_end_idx + 1]
                rebalance_dates.append(next_trading_day)

        rebalance_dates = pd.DatetimeIndex(rebalance_dates)

        strategy_returns = pd.Series(0.0, index=dates)
        current_weights = pd.Series(0.0, index=weights.columns)
        transaction_costs_ts = pd.Series(0.0, index=dates)

        # Calculate returns
        for i in range(len(dates)):
            date = dates[i]

            # Check rebalancing
            if date in rebalance_dates:
                month_end = month_ends[month_ends < date].iloc[-1]
                new_weights = weights.loc[month_end]

                # Calculate turnover
                turnover = (new_weights - current_weights).abs().sum()
                transaction_costs_ts.loc[date] = turnover * self.config.TOTAL_COST_BPS / 10000

                current_weights = new_weights

            # FIX: Consistent Open-to-Open returns for ALL days except last
            if i < len(dates) - 1:
                next_date = dates[i + 1]
                asset_returns = (open_prices.loc[next_date] / open_prices.loc[date] - 1)
                missing = asset_returns.isna() & current_weights.ne(0)
                if missing.any():
                    symbols = ", ".join(missing.index[missing])
                    raise ValueError(f"Missing prices for held symbols on {date.date()}: {symbols}")
                portfolio_return = (current_weights * asset_returns).sum()
                net_return = portfolio_return - transaction_costs_ts.loc[date]
                strategy_returns.loc[date] = net_return
            else:
                # FIX: Last day - return 0 instead of using inconsistent calculation
                # We don't have Open[t+1], so we can't calculate a consistent return
                strategy_returns.loc[date] = 0.0

        return strategy_returns

    def _calculate_performance_metrics(self,
                                       strategy_returns: pd.Series,
                                       benchmark_returns: pd.Series) -> Dict:
        """Calculate comprehensive performance metrics."""
        common_dates = strategy_returns.index.intersection(benchmark_returns.index)
        strat_ret = strategy_returns.loc[common_dates].dropna()
        bench_ret = benchmark_returns.loc[common_dates].dropna()

        common_dates = strat_ret.index.intersection(bench_ret.index)
        strat_ret = strat_ret.loc[common_dates]
        bench_ret = bench_ret.loc[common_dates]

        strat_cum = (1 + strat_ret).cumprod()
        bench_cum = (1 + bench_ret).cumprod()

        ann_factor = self.config.ANNUALIZATION_FACTOR
        years = len(strat_ret) / ann_factor

        metrics = {
            'total_return_strategy': strat_cum.iloc[-1] - 1,
            'total_return_benchmark': bench_cum.iloc[-1] - 1,
            'cagr_strategy': (strat_cum.iloc[-1] ** (1/years)) - 1,
            'cagr_benchmark': (bench_cum.iloc[-1] ** (1/years)) - 1,
            'volatility_strategy': strat_ret.std() * np.sqrt(ann_factor),
            'volatility_benchmark': bench_ret.std() * np.sqrt(ann_factor),
            'downside_dev_strategy': strat_ret[strat_ret < 0].std() * np.sqrt(ann_factor) if len(strat_ret[strat_ret < 0]) > 0 else 0,
            'downside_dev_benchmark': bench_ret[bench_ret < 0].std() * np.sqrt(ann_factor) if len(bench_ret[bench_ret < 0]) > 0 else 0,
            'sharpe_strategy': (strat_ret.mean() * ann_factor) / (strat_ret.std() * np.sqrt(ann_factor)) if strat_ret.std() > 0 else 0,
            'sharpe_benchmark': (bench_ret.mean() * ann_factor) / (bench_ret.std() * np.sqrt(ann_factor)) if bench_ret.std() > 0 else 0,
            'sortino_strategy': (strat_ret.mean() * ann_factor) / (strat_ret[strat_ret < 0].std() * np.sqrt(ann_factor)) if len(strat_ret[strat_ret < 0]) > 0 else 0,
            'sortino_benchmark': (bench_ret.mean() * ann_factor) / (bench_ret[bench_ret < 0].std() * np.sqrt(ann_factor)) if len(bench_ret[bench_ret < 0]) > 0 else 0,
            'max_drawdown_strategy': self._calculate_max_drawdown(strat_cum),
            'max_drawdown_benchmark': self._calculate_max_drawdown(bench_cum),
            'win_rate_strategy': (strat_ret > 0).sum() / len(strat_ret),
            'win_rate_benchmark': (bench_ret > 0).sum() / len(bench_ret),
            'strategy_returns': strat_ret,
            'benchmark_returns': bench_ret,
            'strategy_cumulative': strat_cum,
            'benchmark_cumulative': bench_cum,
        }

        return metrics

    @staticmethod
    def _calculate_max_drawdown(cumulative_returns: pd.Series) -> float:
        """Calculate maximum drawdown."""
        running_max = cumulative_returns.expanding().max()
        drawdown = (cumulative_returns - running_max) / running_max
        return drawdown.min()


# ============================================================================
# VISUALIZATION MODULE
# ============================================================================

class PerformanceVisualizer:
    """Creates visualizations for strategy performance."""

    @staticmethod
    def plot_results(results: Dict, config: StrategyConfig):
        """Generate performance visualization dashboard."""
        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

        strat_cum = results['strategy_cumulative']
        bench_cum = results['benchmark_cumulative']
        strat_ret = results['strategy_returns']
        bench_ret = results['benchmark_returns']

        # 1. Cumulative Returns
        ax1 = fig.add_subplot(gs[0, :])
        ax1.plot(strat_cum.index, strat_cum.values,
                label='Strategy', linewidth=2, color='#2E86AB')
        ax1.plot(bench_cum.index, bench_cum.values,
                label=f'Benchmark ({config.BENCHMARK_TICKER})',
                linewidth=2, color='#A23B72', linestyle='--')
        ax1.set_title('Cumulative Returns', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Cumulative Return', fontsize=12)
        ax1.legend(loc='upper left', fontsize=10)
        ax1.grid(True, alpha=0.3)

        # 2. Rolling Sharpe
        ax2 = fig.add_subplot(gs[1, 0])
        rolling_window = 252
        rolling_sharpe_strat = (
            strat_ret.rolling(rolling_window).mean() /
            strat_ret.rolling(rolling_window).std() *
            np.sqrt(config.ANNUALIZATION_FACTOR)
        )
        rolling_sharpe_bench = (
            bench_ret.rolling(rolling_window).mean() /
            bench_ret.rolling(rolling_window).std() *
            np.sqrt(config.ANNUALIZATION_FACTOR)
        )
        ax2.plot(rolling_sharpe_strat.index, rolling_sharpe_strat.values,
                label='Strategy', linewidth=1.5, color='#2E86AB')
        ax2.plot(rolling_sharpe_bench.index, rolling_sharpe_bench.values,
                label='Benchmark', linewidth=1.5, color='#A23B72', linestyle='--')
        ax2.set_title(f'Rolling Sharpe Ratio ({rolling_window}d)', fontsize=12, fontweight='bold')
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)

        # 3. Drawdown
        ax3 = fig.add_subplot(gs[1, 1])
        strat_running_max = strat_cum.expanding().max()
        strat_dd = (strat_cum - strat_running_max) / strat_running_max
        bench_running_max = bench_cum.expanding().max()
        bench_dd = (bench_cum - bench_running_max) / bench_running_max

        ax3.fill_between(strat_dd.index, strat_dd.values, 0,
                        alpha=0.3, color='#2E86AB', label='Strategy')
        ax3.fill_between(bench_dd.index, bench_dd.values, 0,
                        alpha=0.3, color='#A23B72', label='Benchmark')
        ax3.set_title('Drawdown', fontsize=12, fontweight='bold')
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3)

        # 4. Monthly Returns Heatmap
        ax4 = fig.add_subplot(gs[1, 2])
        monthly_returns = strat_ret.resample('ME').apply(lambda x: (1 + x).prod() - 1)
        monthly_returns_pivot = monthly_returns.groupby([
            monthly_returns.index.year,
            monthly_returns.index.month
        ]).first().unstack()

        if len(monthly_returns_pivot) > 0:
            sns.heatmap(monthly_returns_pivot * 100, annot=True, fmt='.1f',
                       cmap='RdYlGn', center=0, ax=ax4)
            ax4.set_title('Monthly Returns (%)', fontsize=12, fontweight='bold')

        # 5. Return Distribution
        ax5 = fig.add_subplot(gs[2, 0])
        ax5.hist(strat_ret * 100, bins=50, alpha=0.7, color='#2E86AB',
                label='Strategy', density=True)
        ax5.hist(bench_ret * 100, bins=50, alpha=0.7, color='#A23B72',
                label='Benchmark', density=True)
        ax5.set_title('Return Distribution', fontsize=12, fontweight='bold')
        ax5.legend(fontsize=9)
        ax5.grid(True, alpha=0.3)

        # 6. Metrics Table
        ax6 = fig.add_subplot(gs[2, 1:])
        ax6.axis('tight')
        ax6.axis('off')

        metrics_data = [
            ['Metric', 'Strategy', 'Benchmark', 'Difference'],
            ['Total Return',
             f"{results['total_return_strategy']*100:.2f}%",
             f"{results['total_return_benchmark']*100:.2f}%",
             f"{(results['total_return_strategy']-results['total_return_benchmark'])*100:.2f}%"],
            ['CAGR',
             f"{results['cagr_strategy']*100:.2f}%",
             f"{results['cagr_benchmark']*100:.2f}%",
             f"{(results['cagr_strategy']-results['cagr_benchmark'])*100:.2f}%"],
            ['Volatility',
             f"{results['volatility_strategy']*100:.2f}%",
             f"{results['volatility_benchmark']*100:.2f}%",
             f"{(results['volatility_strategy']-results['volatility_benchmark'])*100:.2f}%"],
            ['Sharpe Ratio',
             f"{results['sharpe_strategy']:.2f}",
             f"{results['sharpe_benchmark']:.2f}",
             f"{results['sharpe_strategy']-results['sharpe_benchmark']:.2f}"],
            ['Max Drawdown',
             f"{results['max_drawdown_strategy']*100:.2f}%",
             f"{results['max_drawdown_benchmark']*100:.2f}%",
             f"{(results['max_drawdown_strategy']-results['max_drawdown_benchmark'])*100:.2f}%"],
        ]

        table = ax6.table(cellText=metrics_data, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2)

        plt.suptitle(f'Strategy Performance\n'
                    f'Period: {config.START_DATE} to {config.END_DATE}',
                    fontsize=16, fontweight='bold', y=0.98)

        plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function."""
    config = StrategyConfig()

    backtester = SMOMBacktester(config)
    results = backtester.run_backtest()

    print("\n" + "="*80)
    print("PERFORMANCE SUMMARY")
    print("="*80)
    print(f"Strategy Total Return:    {results['total_return_strategy']*100:>8.2f}%")
    print(f"Benchmark Total Return:   {results['total_return_benchmark']*100:>8.2f}%")
    print("-"*80)
    print(f"Strategy CAGR:            {results['cagr_strategy']*100:>8.2f}%")
    print(f"Benchmark CAGR:           {results['cagr_benchmark']*100:>8.2f}%")
    print("-"*80)
    print(f"Strategy Sharpe Ratio:    {results['sharpe_strategy']:>8.2f}")
    print(f"Benchmark Sharpe Ratio:   {results['sharpe_benchmark']:>8.2f}")
    print("-"*80)
    print(f"Strategy Max Drawdown:    {results['max_drawdown_strategy']*100:>8.2f}%")
    print(f"Benchmark Max Drawdown:   {results['max_drawdown_benchmark']*100:>8.2f}%")
    print("="*80)

    PerformanceVisualizer.plot_results(results, config)

    return results


if __name__ == "__main__":
    results = main()
